# momentum-buffer-update composite — cx9: canonical SGD-momentum step: v = mu*v + g, then p -= lr * v

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `momentum-buffer-update`, `inplace-param-update`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "momentum-buffer-update"
DD_ATOM_IDS = ["momentum-buffer-update", "inplace-param-update"]
DD_SUBTOPICS = ["Optimizer: Momentum buffer", "PyTorch: In-place param update"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The textbook SGD-momentum step has two lines of work per parameter:

1. **`momentum-buffer-update`** — exponentially-decayed accumulator of past gradients: `v <- mu * v + g`. (Some impls scale `g` by `1 - mu` first; PyTorch's default uses the form above, matching Sutskever et al.)
2. **`inplace-param-update`** — descend along the velocity (not the raw gradient): `p <- p - lr * v`. Has to be in-place on `p.data` so the caller (and the next forward pass) sees the new weights.

**Anatomy.**
```python
for p, v in zip(params, velocity_buffers):
    g = p.grad
    v.mul_(mu).add_(g)               # momentum-buffer-update (in-place)
    p.data.add_(v, alpha=-lr)        # inplace-param-update
```

**Why both atoms together.** Momentum buys you BOTH bigger steps in low-curvature directions (velocity accumulates) AND damping in oscillatory directions (gradient sign flips cancel). Neither benefit shows up if the param isn't actually updated — and the param has to move by the VELOCITY, not the raw gradient. Pairing the two atoms is what makes the optimizer 'SGD with momentum' rather than 'plain SGD'.

### Composite Exercise — canonical SGD-momentum step: v = mu*v + g, then p -= lr * v

**Atoms exercised together**: `momentum-buffer-update`, `inplace-param-update`

Implement `cx9_sgd_momentum(params, grads, velocity_buffers, lr, mu)`.

Apply ONE step of SGD with momentum to every triple `(p, g, v)`:

1. Update the velocity buffer in place: `v <- mu * v + g`. Use `v.mul_(mu).add_(g)` (or `v.copy_(mu*v + g)`). The buffer object must persist across calls so subsequent steps accumulate.
2. Update the parameter in place using the NEW velocity: `p.data.add_(v, alpha=-lr)`. The param must move by `-lr * v` (the velocity), NOT `-lr * g` (the raw gradient).

Return `None`. Test cross-checks against `torch.optim.SGD(momentum=mu)` over two consecutive steps. Step 2 is the discriminator: if you used `g` instead of `v` in the param update, step 1 may still look right (with zero-init buffer, `v == g`) but step 2 will diverge because the velocity now carries history.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx9_sgd_momentum(params, grads, velocity_buffers, lr, mu):
    """v <- mu*v + g; p.data -= lr * v. In place. Returns None."""
    raise NotImplementedError

def _test_cx9():
    t.manual_seed(0)
    shapes = [(4,), (3, 2)]
    params = [t.nn.Parameter(t.randn(s)) for s in shapes]
    velocity = [t.zeros_like(p) for p in params]
    ref_params = [t.nn.Parameter(p.data.clone()) for p in params]
    opt_ref = t.optim.SGD(ref_params, lr=0.05, momentum=0.95)

    buf_ptrs_before = [b.data_ptr() for b in velocity]
    param_ptrs_before = [p.data.data_ptr() for p in params]

    # === Step 1 ===
    g1 = [t.randn(s) for s in shapes]
    for rp, g in zip(ref_params, g1):
        rp.grad = g.clone()
    opt_ref.step()
    cx9_sgd_momentum(params, g1, velocity, lr=0.05, mu=0.95)

    for i in range(len(params)):
        assert t.allclose(params[i].data, ref_params[i].data, atol=1e-6), (
            f'step 1 params[{i}] mismatch vs torch.optim.SGD'
        )
        ref_v = opt_ref.state[ref_params[i]]['momentum_buffer']
        assert t.allclose(velocity[i], ref_v, atol=1e-6), (
            f'step 1 velocity[{i}] mismatch'
        )
        assert velocity[i].data_ptr() == buf_ptrs_before[i], (
            f'velocity[{i}] storage reallocated — buffer must update in place'
        )
        assert params[i].data.data_ptr() == param_ptrs_before[i], (
            f'params[{i}].data storage reallocated'
        )

    # === Step 2 — the discriminator. ===
    g2 = [t.randn(s) for s in shapes]
    for rp, g in zip(ref_params, g2):
        rp.grad = g.clone()
    opt_ref.step()
    cx9_sgd_momentum(params, g2, velocity, lr=0.05, mu=0.95)
    for i in range(len(params)):
        assert t.allclose(params[i].data, ref_params[i].data, atol=1e-6), (
            f'step 2 params[{i}] mismatch — likely used g instead of v in param update '
            f'(works at step 1 because v=g when buffer starts at 0)'
        )

    # === Sanity: mu=0 collapses to plain SGD. ===
    p3 = t.nn.Parameter(t.tensor([5.0, -3.0]))
    v3 = t.zeros_like(p3)
    p3_before = p3.data.clone()
    g3 = [t.tensor([1.0, 1.0])]
    cx9_sgd_momentum([p3], g3, [v3], lr=0.1, mu=0.0)
    assert t.allclose(p3.data, p3_before - 0.1 * g3[0], atol=1e-6), 'mu=0 should give plain SGD'
    assert t.allclose(v3, g3[0], atol=1e-6), 'mu=0: buffer should equal g'

    # === Sanity: lr=0 with mu>0 leaves params unchanged but BUILDS velocity. ===
    p4 = t.nn.Parameter(t.tensor([10.0]))
    v4 = t.zeros_like(p4)
    p4_before = p4.data.clone()
    cx9_sgd_momentum([p4], [t.tensor([1.0])], [v4], lr=0.0, mu=0.9)
    assert t.allclose(p4.data, p4_before), 'lr=0: param should not move'
    assert t.allclose(v4, t.tensor([1.0])), 'lr=0: velocity should still accumulate'
    _dd_passed.add('cx9')

_test_cx9()

<details><summary>Show solution — cx9</summary>

```python
def cx9_sgd_momentum(params, grads, velocity_buffers, lr, mu):
    for p, g, v in zip(params, grads, velocity_buffers):
        # Atom A (momentum-buffer-update): v <- mu*v + g, IN PLACE.
        v.mul_(mu).add_(g)
        # Atom B (inplace-param-update): step ALONG THE VELOCITY, not the raw gradient.
        p.data.add_(v, alpha=-lr)
    return None
```

`v.mul_(mu).add_(g)` does the right thing in one fused pass per element — equivalent to `v.copy_(mu*v + g)` but with no temporary. The most common bug is writing `p.data.add_(g, alpha=-lr)` instead of `p.data.add_(v, alpha=-lr)` — that's just plain SGD, and step 1 may even look correct because `v == g` at step 1 (zero-init buffer). Step 2 diverges immediately. PyTorch's actual SGD has a `dampening` parameter that scales `g` by `(1 - dampening)` before adding to `v` — default `dampening=0`, so we ignore it here.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx9',
        'subtopics': ["Optimizer: Momentum buffer", "PyTorch: In-place param update"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()